# Multi-Agent Research Assistant — Walkthrough

**Author:** Sabal Nemkul  
**Module:** COS6031-D Applied AI Professional — Element 3 portfolio  
**Project tier:** Complex (AI agents, LangGraph)

## What this notebook does

Walks through the multi-agent research pipeline step by step:

1. Inspect the `ResearchState` schema
2. Run each agent in isolation to see what they produce
3. Build the full LangGraph pipeline and run it end-to-end
4. Visualise the graph topology

For a polished interactive UI use `streamlit run src/app.py` instead.

## 1. Setup

Set both `ANTHROPIC_API_KEY` and `TAVILY_API_KEY` (e.g. via a `.env` file).

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

sys.path.insert(0, str(Path('..').resolve() / 'src'))

assert os.getenv('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY'
assert os.getenv('TAVILY_API_KEY'),    'Set TAVILY_API_KEY'

## 2. The shared state

Every agent reads from and writes to a single `ResearchState` TypedDict. This is the central contract that lets agents work together without explicit coordination.

In [ ]:
from state import ResearchState, SubQuery, SearchResult
import inspect
print(inspect.getsource(ResearchState))

## 3. The Planner agent in isolation

In [ ]:
from agents import planner_node

question = 'What are the main risks of deploying LLMs in clinical decision support?'
result = planner_node({'question': question, 'log': []})

for i, sq in enumerate(result['sub_queries'], 1):
    print(f"{i}. {sq['question']}")
    print(f"   Rationale: {sq['rationale']}\n")

## 4. The Researcher agent in isolation

Takes the sub-queries from the Planner and runs a Tavily web search for each one.

In [ ]:
from agents import researcher_node

research_result = researcher_node({
    'question': question,
    'sub_queries': result['sub_queries'],
    'log': [],
})

print(f"Total results: {len(research_result['search_results'])}\n")
for i, r in enumerate(research_result['search_results'][:5], 1):
    print(f"[{i}] {r['title']}\n    {r['url']}\n    {r['snippet'][:150]}…\n")

## 5. Run the full pipeline

Wire everything into a graph and run end-to-end.

In [ ]:
from graph import run_research

final_state = run_research(
    'What is the current state of edge AI deployment in UK retail?',
    save_output=False,
)

print('\n=== Final report ===\n')
print(final_state['final_report'])

## 6. Visualise the graph

LangGraph compiles graphs into actual state machines we can render.

In [ ]:
from graph import build_graph
from IPython.display import Image, display

g = build_graph()
display(Image(g.get_graph().draw_mermaid_png()))

## 7. Reflection

**What worked.**
- The state-driven design kept the agent code very small. Each node is one focused function.
- Splitting the work into Planner/Researcher/Synthesiser produced visibly better reports than asking a single LLM call to do everything.
- LangGraph's clear separation of state schema, nodes, and edges made debugging much easier — when an agent misbehaved, I could see exactly what state it received.

**What does not work yet.**
- Linear graph only — no "are these sources sufficient?" loop back to the Researcher.
- No deduplication or quality filtering of retrieved sources.
- No structured output schema for the Synthesiser — relies on Markdown convention.
- No evaluation framework.

**Connection to other portfolio work.**
- This project's architectural pattern (specialised stages, shared state, cheap stages filtering for expensive ones) is the same pattern as the four-stage detection pipeline in my Industrial AI Project. The abstraction level is different — agents instead of OpenCV stages — but the design principle is identical.
- The honest-evaluation discipline I applied to my Element 2 (engaging critically with the 97.6%→35% generalisation gap) carries over here in the Limitations section: I'd rather flag what's missing than pretend the v1 is finished.